In [130]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [131]:
import pandas as pd
import numpy as np
from pyMyriad import *
from pyMyriad.tabular import flatten, tabulate
from pyMyriad.listing import gt_table, simple_table

In [132]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "Education": np.random.normal(0, 1, 1000),
  "Employed": np.random.choice([0, 1], 1000)
})

In [133]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
efun = lambda df: np.mean(df.Education)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .summarize_by(m = mfun, n = nfun)\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age Group", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun, label = "Income analysis")\
  .analyze_by(m = efun, label = "Education analysis")

res = atree.run(df)


# Tabulation with simple_table()

The `simple_table()` function provides a lightweight way to display analysis results as a pandas DataFrame without requiring the great-tables package.

In [134]:
# Basic simple_table - shows only analysis results
simple_table(res)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
3,Gender,F,--,--,--,--,m,49990.728784
3,,,--,--,--,--,n,14670.133814
8,,,Benin Y/N,False,Age Group,age40,m,50146.961408
8,,,,,,,n,14806.586378
9,,,,,,,m,-0.02593
11,,,,,,age60,m,49451.218844
11,,,,,,,n,12761.802668
12,,,,,,,m,-0.158014
16,,,,True,,age40,m,49371.835489
16,,,,,,,n,14627.835696


## simple_table with pivot

You can pivot by a split variable to show results across columns:

In [135]:
# Pivot by Gender to show results side-by-side
simple_table(res, by="df.Gender")

pivot_lvl,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,F,M
0,--,--,--,--,m,49990.728784,49905.929121
1,--,--,--,--,n,14670.133814,14745.41975
2,Benin Y/N,False,Age Group,age40,m,-0.02593,-0.091774
3,,,,,m,50146.961408,49180.279748
4,,,,,n,14806.586378,14610.377603
5,,,,age60,m,-0.158014,-0.24099
6,,,,,m,49451.218844,47699.315403
7,,,,,n,12761.802668,15205.049586
8,,True,,age40,m,0.124103,0.098963
9,,,,,m,49371.835489,49570.56401


## simple_table with all nodes

Include non-analysis nodes (splits and summaries) by setting `include_non_analysis=True`:

In [136]:
# Show all nodes including splits
simple_table(res, include_non_analysis=True).head(10)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
0,--,--,--,--,--,--,None,None
1,Gender,--,--,--,--,--,None,None
2,,F,--,--,--,--,None,None
3,,,--,--,--,--,m,49990.728784
3,,,--,--,--,--,n,14670.133814
4,,,Benin Y/N,--,--,--,None,None
5,,,,False,--,--,None,None
6,,,,,Age Group,--,None,None
7,,,,,,age40,None,None
8,,,,,,,m,50146.961408


## simple_table without duplicate suppression

By default, duplicate values in hierarchy columns are suppressed for cleaner display. You can disable this:

In [137]:
# Show all duplicate values
simple_table(res, suppress_duplicates=False).head()

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
3,Gender,F,--,--,--,--,m,49990.728784
3,Gender,F,--,--,--,--,n,14670.133814
8,Gender,F,Benin Y/N,False,Age Group,age40,m,50146.961408
8,Gender,F,Benin Y/N,False,Age Group,age40,n,14806.586378
9,Gender,F,Benin Y/N,False,Age Group,age40,m,-0.02593


# Tabulation with gt_table()

The `gt_table()` function creates beautifully formatted tables using the great-tables package. It provides more styling options and is ideal for reports and presentations.

In [138]:
# Basic gt_table with title
gt_table(res, title="Analysis Results").show()

Analysis Results 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 -- 
 -- 
 -- 
 -- 
 m 
 49990.728783924525 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14670.133814185387 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 50146.96140811971 
 
 
 
 
 
 
 
 
 n 
 14806.586378069254 
 
 
 
 
 
 
 
 
 m 
 -0.025930229846427334 
 
 
 
 
 
 
 
 age60 
 m 
 49451.218844387404 
 
 
 
 
 
 
 
 
 n 
 12761.802667913413 
 
 
 
 
 
 
 
 
 m 
 -0.1580144145684637 
 
 
 
 
 
 True 
 
 age40 
 m 
 49371.835489343975 
 
 
 
 
 
 
 
 
 n 
 14627.835696468155 
 
 
 
 
 
 
 
 
 m 
 0.12410339085490052 
 
 
 
 
 
 
 
 age60 
 m 
 50195.727253251134 
 
 
 
 
 
 
 
 
 n 
 14611.851568968785 
 
 
 
 
 
 
 
 
 m 
 0.147029549908047 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 m 
 49905.929121388894 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14745.419750135508 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 49180.27974828969 
 
 
 
 
 
 
 
 
 n 
 14610.377603314017 
 
 
 
 
 
 
 
 
 m 
 -0.09177351080983902 
 
 
 
 
 
 
 
 age60 
 m 
 47699.3154030276 
 
 
 
 
 
 
 
 
 n 
 15205.049586134182 
 
 
 
 
 
 
 
 
 m 
 -0.24098982915488953 
 
 
 
 
 
 True 
 
 age40 
 m 
 49570.5640097929 
 
 
 
 
 
 
 
 
 n 
 15023.589510946611 
 
 
 
 
 
 
 
 
 m 
 0.09896286650221331 
 
 
 
 
 
 
 
 age60 
 m 
 50871.22374887286 
 
 
 
 
 
 
 
 
 n 
 14196.945292175289 
 
 
 
 
 
 
 
 
 m 
 0.0037220870319372456

## gt_table with pivot

Pivot by a split variable:

In [139]:
# Pivot by Gender with custom title and subtitle
gt_table(
    res, 
    by="df.Gender", 
    title="Gender Comparison",
    subtitle="Income and Education Analysis by Gender"
).show()

Gender Comparison 
 
 
 Income and Education Analysis by Gender 
 
 
 
 Hierarchy 
 
 Statistic 
 F 
 M 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49990.728783924525 
 49905.929121388894 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14670.133814185387 
 14745.419750135508 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 -0.025930229846427334 
 -0.09177351080983902 
 
 
 
 
 
 
 m 
 50146.96140811971 
 49180.27974828969 
 
 
 
 
 
 
 n 
 14806.586378069254 
 14610.377603314017 
 
 
 
 
 
 age60 
 m 
 -0.1580144145684637 
 -0.24098982915488953 
 
 
 
 
 
 
 m 
 49451.218844387404 
 47699.3154030276 
 
 
 
 
 
 
 n 
 12761.802667913413 
 15205.049586134182 
 
 
 
 True 
 
 age40 
 m 
 0.12410339085490052 
 0.09896286650221331 
 
 
 
 
 
 
 m 
 49371.835489343975 
 49570.5640097929 
 
 
 
 
 
 
 n 
 14627.835696468155 
 15023.589510946611 
 
 
 
 
 
 age60 
 m 
 0.147029549908047 
 0.0037220870319372456 
 
 
 
 
 
 
 m 
 50195.727253251134 
 50871.22374887286 
 
 
 
 
 
 
 n 
 14611.851568968785 
 14196.945292175289

## gt_table with all nodes and custom formatting

Include non-analysis nodes and customize decimal places:

In [140]:
# Show all nodes with 2 decimal places
gt_table(
    res, 
    title="Complete Analysis Tree",
    include_non_analysis=True,
    decimals=2
).show()

Complete Analysis Tree 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 Gender 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 F 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49990.728783924525 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14670.133814185387 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 50146.96140811971 
 
 
 
 
 
 
 
 
 n 
 14806.586378069254 
 
 
 
 
 
 
 
 
 m 
 -0.025930229846427334 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 49451.218844387404 
 
 
 
 
 
 
 
 
 n 
 12761.802667913413 
 
 
 
 
 
 
 
 
 m 
 -0.1580144145684637 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49371.835489343975 
 
 
 
 
 
 
 
 
 n 
 14627.835696468155 
 
 
 
 
 
 
 
 
 m 
 0.12410339085490052 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 50195.727253251134 
 
 
 
 
 
 
 
 
 n 
 14611.851568968785 
 
 
 
 
 
 
 
 
 m 
 0.147029549908047 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 m 
 49905.929121388894 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 n 
 14745.419750135508 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49180.27974828969 
 
 
 
 
 
 
 
 
 n 
 14610.377603314017 
 
 
 
 
 
 
 
 
 m 
 -0.09177351080983902 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 47699.3154030276 
 
 
 
 
 
 
 
 
 n 
 15205.049586134182 
 
 
 
 
 
 
 
 
 m 
 -0.24098982915488953 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49570.5640097929 
 
 
 
 
 
 
 
 
 n 
 15023.589510946611 
 
 
 
 
 
 
 
 
 m 
 0.09896286650221331 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 50871.22374887286 
 
 
 
 
 
 
 
 
 n 
 14196.945292175289 
 
 
 
 
 
 
 
 
 m 
 0.0037220870319372456

# Pivoting Statistics as Columns

The `pivot_statistics` parameter allows you to display statistics as columns instead of rows, creating a wider, more compact table format.

In [141]:
# Pivot statistics into columns - simple_table
simple_table(res, pivot_statistics=True)

statistics,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,m,n
0,Gender,F,--,--,--,--,49990.728784,14670.133814
1,,M,--,--,--,--,49905.929121,14745.41975
2,,F,Benin Y/N,False,Age Group,age40,-0.02593,NaN
3,,,,,,,50146.961408,14806.586378
4,,,,,,age60,-0.158014,NaN
5,,,,,,,49451.218844,12761.802668
6,,,,True,,age40,0.124103,NaN
7,,,,,,,49371.835489,14627.835696
8,,,,,,age60,0.14703,NaN
9,,,,,,,50195.727253,14611.851569


## Combining pivot_statistics with by

You can combine both pivots to create a matrix-style table with split groups and statistics as columns:

In [142]:
# Combine both pivots - shows Gender groups with statistics as columns
simple_table(res, by="df.Gender", pivot_statistics=True)

,_Level_0,_Level_1,_Level_2,_Level_3,F||m,F||n,M||m,M||n
0,--,--,--,--,49990.728784,14670.133814,49905.929121,14745.41975
1,Benin Y/N,False,Age Group,age40,-0.02593,NaN,-0.091774,NaN
2,,,,,50146.961408,14806.586378,49180.279748,14610.377603
3,,,,age60,-0.158014,NaN,-0.24099,NaN
4,,,,,49451.218844,12761.802668,47699.315403,15205.049586
5,,True,,age40,0.124103,NaN,0.098963,NaN
6,,,,,49371.835489,14627.835696,49570.56401,15023.589511
7,,,,age60,0.14703,NaN,0.003722,NaN
8,,,,,50195.727253,14611.851569,50871.223749,14196.945292


## gt_table with pivot_statistics

The same feature works with `gt_table()` for beautifully formatted output:

In [143]:
# gt_table with statistics as columns
gt_table(
    res,
    pivot_statistics=True,
    title="Analysis with Statistics as Columns"
).show()

Analysis with Statistics as Columns 
 
 
 
 Hierarchy 
 
 m 
 n 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 -- 
 -- 
 -- 
 -- 
 49990.728783924525 
 14670.133814185387 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 49905.929121388894 
 14745.419750135508 
 
 
 
 F 
 Benin Y/N 
 False 
 Age Group 
 age40 
 -0.025930229846427334 
 
 
 
 
 
 
 
 
 
 50146.96140811971 
 14806.586378069254 
 
 
 
 
 
 
 
 age60 
 -0.1580144145684637 
 
 
 
 
 
 
 
 
 
 49451.218844387404 
 12761.802667913413 
 
 
 
 
 
 True 
 
 age40 
 0.12410339085490052 
 
 
 
 
 
 
 
 
 
 49371.835489343975 
 14627.835696468155 
 
 
 
 
 
 
 
 age60 
 0.147029549908047 
 
 
 
 
 
 
 
 
 
 50195.727253251134 
 14611.851568968785 
 
 
 
 M 
 
 False 
 
 age40 
 -0.09177351080983902 
 
 
 
 
 
 
 
 
 
 49180.27974828969 
 14610.377603314017 
 
 
 
 
 
 
 
 age60 
 -0.24098982915488953 
 
 
 
 
 
 
 
 
 
 47699.3154030276 
 15205.049586134182 
 
 
 
 
 
 True 
 
 age40 
 0.09896286650221331 
 
 
 
 
 
 
 
 
 
 49570.5640097929 
 15023.589510946611 
 
 
 
 
 
 
 
 age60 
 0.0037220870319372456 
 
 
 
 
 
 
 
 
 
 50871.22374887286 
 14196.945292175289

In [144]:
# gt_table combining both pivots
gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    title="Gender Comparison - Matrix View",
    subtitle="Each group-statistic combination as a column"
).show()

Gender Comparison - Matrix View 
 
 
 Each group-statistic combination as a column 
 
 
 
 Hierarchy 
 
 
 F 
 
 
 M 
 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 m 
 n 
 m 
 n 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 49990.728783924525 
 14670.133814185387 
 49905.929121388894 
 14745.419750135508 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 -0.025930229846427334 
 
 -0.09177351080983902 
 
 
 
 
 
 
 
 50146.96140811971 
 14806.586378069254 
 49180.27974828969 
 14610.377603314017 
 
 
 
 
 
 age60 
 -0.1580144145684637 
 
 -0.24098982915488953 
 
 
 
 
 
 
 
 49451.218844387404 
 12761.802667913413 
 47699.3154030276 
 15205.049586134182 
 
 
 
 True 
 
 age40 
 0.12410339085490052 
 
 0.09896286650221331 
 
 
 
 
 
 
 
 49371.835489343975 
 14627.835696468155 
 49570.5640097929 
 15023.589510946611 
 
 
 
 
 
 age60 
 0.147029549908047 
 
 0.0037220870319372456 
 
 
 
 
 
 
 
 50195.727253251134 
 14611.851568968785 
 50871.22374887286 
 14196.945292175289

In [148]:
print(res)

Data Tree
  Split: df.Gender
    └- F
        analysis: 
        └- m: 49990.728783924525
        └- n: 14670.133814185387
      Split: Benin Y/N
        └- False
          Split: Age Group
            └- age40
                analysis: Income analysis
                └- m: 50146.96140811971
                └- n: 14806.586378069254
                analysis: Education analysis
                └- m: -0.025930229846427334
            └- age60
                analysis: Income analysis
                └- m: 49451.218844387404
                └- n: 12761.802667913413
                analysis: Education analysis
                └- m: -0.1580144145684637
        └- True
          Split: Age Group
            └- age40
                analysis: Income analysis
                └- m: 49371.835489343975
                └- n: 14627.835696468155
                analysis: Education analysis
                └- m: 0.12410339085490052
            └- age60
                analysis: Income analysis
       

In [154]:
rr = format_statistics(res, format_map ={"Income analysis": "{m:.0f} ∞ {n}"}, stat_name="m", inplace=False, remove_original=True)
print(rr)

Data Tree
  Split: df.Gender
    └- F
        analysis: 
        └- m: 49990.728783924525
        └- n: 14670.133814185387
      Split: Benin Y/N
        └- False
          Split: Age Group
            └- age40
                analysis: Income analysis
                └- m: 50147 ∞ 14806.586378069254
                analysis: Education analysis
                └- m: -0.025930229846427334
            └- age60
                analysis: Income analysis
                └- m: 49451 ∞ 12761.802667913413
                analysis: Education analysis
                └- m: -0.1580144145684637
        └- True
          Split: Age Group
            └- age40
                analysis: Income analysis
                └- m: 49372 ∞ 14627.835696468155
                analysis: Education analysis
                └- m: 0.12410339085490052
            └- age60
                analysis: Income analysis
                └- m: 50196 ∞ 14611.851568968785
                analysis: Education analysis
           